In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from datetime import datetime, timedelta
import pandas as pd
import h5py 


In [ ]:


# Define el directorio base
base_directory_cc = '/data/data4/veronica-scratch-rainier/swarm_august2023/results_CC_TMA/'

# Nombre variable de la carpeta
folder_name = 'CC_5sec-tem_2023-08-25_00.00-2023-08-31_00.00/' 
# Ruta completa
full_path = os.path.join(base_directory_cc, folder_name)

# Crea la carpeta si no existe
if not os.path.exists(full_path):
    os.makedirs(full_path)
print(full_path)

def mad_func_shelly(arr):
    """Desviación Absoluta Mediana: Usando la formulación en Li y Zhan 2018."""
    med = np.median(arr)
    return np.median(np.abs(arr - med))

def calculate_detection_sig(folder_data):
    """Calcula la significancia de la detección para los datos de una carpeta dada."""
    median = np.median(folder_data)
    mad = mad_func_shelly(folder_data)
    detection_sig = (folder_data - median) / mad
    return detection_sig, mad

def plot_histogram(ax, detection_sig, folder_name):
    """Grafica el histograma de la significancia de detección."""
    ax.hist(detection_sig, bins=1000, range=(0, 500), alpha=0.75, color='#1f77b4', edgecolor='black', linewidth=0.5)
    ax.set_yscale('log')  # Establece el eje y en escala logarítmica
    ax.set_xlabel('Detection Significance', fontsize=14)
    ax.set_ylabel('Counts (log scale)', fontsize=14)
    ax.grid(True, which="both", ls="--", linewidth=0.5)
    ax.set_title(f'Histogram of Detection Significance for {folder_name}', fontsize=16)

def plot_time_series(ax, time_utc, folder_data, mad, folder_name):
    """Grafica la serie temporal del valor de correlación sobre MAD."""
    ax.plot(time_utc, folder_data / mad, label=f'Template {folder_name}', color='#ff7f0e', linestyle='-', linewidth=1.5)
    ax.set_title(f'Template {folder_name}', fontsize=16)
    ax.set_xlabel('Time (UTC)', fontsize=14)
    ax.set_ylabel('Detection Significance', fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, which="both", ls="--", linewidth=0.5)
    ax.xaxis.set_major_formatter(DateFormatter('%Y-%m-%d %H:%M:%S'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    #adding the new events to the timeseries -*
    for date in unmatched_dates:
        ax.axvline(x=date, color = 'red', linestyle = '--', linewidth = 1)
        ax.plot(date,0.05,'r*',markersize = 10)

def convert_timestamps_to_utc(timestamps):
    
    """Convierte los timestamps en microsegundos a tiempos en UTC."""
    start_time = datetime(1970, 1, 1)  # Epoch time
    return [start_time + timedelta(microseconds=int(ts)) for ts in timestamps]

def read_dataframe(file_path):
    
    """Read txt file and look for the not matches"""
    
    df = pd.read_csv(file_path)
    unmatched_df = df[df['Match-PNSN'] == 'No']
    unmatched_dates = pd.to_datetime(unmatched_df['Unique Detection Time (UTC)'], format = '%Y-%m-%d %H:%M:%S')
    
    return unmatched_dates

def process_single_folder(full_path, folder_name, output_plot_directory, h5_file_path,unmatched_dates):
    
    folder_path = os.path.join(full_path, folder_name)
    try:
        npy_files = [np.load(os.path.join(folder_path, file)) for file in os.listdir(folder_path) if file.endswith('.npy')]
        folder_data = np.concatenate(npy_files, axis=0)

        # Leer los timestamps del archivo .h5
        with h5py.File(h5_file_path, 'r') as h5_file:
            timestamps = np.array(h5_file['timestamps'])
            timestamps = timestamps.astype(int)  # Convertir explícitamente a int
            time_utc = convert_timestamps_to_utc(timestamps)
    except Exception as e:
        print(f"Error processing folder {folder_path}: {e}")
        return
    
    detection_sig, mad = calculate_detection_sig(folder_data)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [3, 2]})

    plot_time_series(ax1, time_utc[:len(folder_data)], folder_data, mad, folder_name)
    plot_histogram(ax2, detection_sig, folder_name)

    fig.tight_layout()
    plot_filename = os.path.join(output_plot_directory, f'plot_{folder_name}.png')
    plt.savefig(plot_filename, dpi=300)
    plt.show()
    #plt.close(fig)

# Uso de ejemplo
output_plot_directory = os.path.join(full_path, 'plots')
if not os.path.exists(output_plot_directory):
    os.makedirs(output_plot_directory)

folder_name = '2023-08-27_10-10-23.770'  # Reemplaza con el nombre de la carpeta que deseas procesar
h5_file_path = '/data/data4/veronica-scratch-rainier/swarm_august2023/results_CC_TMA/h5_files_timestamps/timestamps_2023-08-25_2023-08-31_00.00.00.h5'  # Ruta al archivo .h5 que contiene los timestamps
unmatched_file_path = '/home/velgueta/notebooks/project_Mt-Rainier_DAS/txtfiles/template_5sec_csv_results-2023-08-25_00.00-2023-08-31_00.00/resulting_detections.csv'

unmatched_dates = read_dataframe(unmatched_file_path)

process_single_folder(full_path, folder_name, output_plot_directory, h5_file_path, unmatched_dates)


In [ ]:


# Define el directorio base
base_directory_cc = '/data/data4/veronica-scratch-rainier/swarm_august2023/results_CC_TMA/'

# Nombre variable de la carpeta
folder_name = 'CC_5sec-tem_2023-08-25_00.00-2023-08-31_00.00/' 
# Complete path
full_path = os.path.join(base_directory_cc, folder_name)

# Create the folder if it doesn't exist
if not os.path.exists(full_path):
    os.makedirs(full_path)
print(full_path)

def mad_func_shelly(arr):
    """Median Absolute Deviation: Using the formulation in Li and Zhan 2018."""
    med = np.median(arr)
    return np.median(np.abs(arr - med))

def calculate_detection_sig(folder_data):
    """Calculates the detection significance for the given folder data."""
    median = np.median(folder_data)
    mad = mad_func_shelly(folder_data)
    detection_sig = (folder_data - median) / mad
    return detection_sig, mad

def plot_histogram(ax, detection_sig, folder_name):
    """Plots the histogram of detection significance."""
    ax.hist(detection_sig, bins=1000, range=(0, 750), alpha=0.75, color='#1f77b4', edgecolor='black', linewidth=0.5)
    ax.set_yscale('log')  # Set y-axis to logarithmic scale
    ax.set_xlabel('Detection Significance', fontsize=14)
    ax.set_ylabel('Counts (log scale)', fontsize=14)
    ax.axvline(x=18, color = 'red', linestyle = ':', linewidth = 2)
    ax.grid(True, which="both", ls="--", linewidth=0.5)
    ax.set_title(f'Histogram of Detection Significance for {folder_name}', fontsize=16)

def plot_time_series(ax, time_utc, folder_data, mad, folder_name, unmatched_dates):
    """Plots the time series of correlation value over MAD."""
    ax.plot(time_utc, folder_data / mad, label=f'Template {folder_name}', color='#ff7f0e', linestyle='-', linewidth=1.5)
    #ax.set_title(f'Template {folder_name}', fontsize=16)
    ax.set_xlabel('Time (UTC)', fontsize=14)
    ax.set_ylabel('Detection Significance', fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, which="both", ls="--", linewidth=0.5)
    ax.xaxis.set_major_formatter(DateFormatter('%Y-%m-%d %H:%M:%S'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Adding unmatched dates to the time series plot
    for date in unmatched_dates:
        #ax.axvline(x=date, color='red', linestyle='--', linewidth=1)
        ax.plot(date, 0, 'b*', markersize=10)  # Red star at y=0

def convert_timestamps_to_utc(timestamps):
    """Converts timestamps in microseconds to UTC times."""
    start_time = datetime(1970, 1, 1)  # Epoch time
    return [start_time + timedelta(microseconds=int(ts)) for ts in timestamps]

def read_dataframe(file_path):
    
    """Read txt file and look for the not matches"""
    
    df = pd.read_csv(file_path)
    unmatched_df = df[df['Match-PNSN'] == 'No']
    unmatched_dates = pd.to_datetime(unmatched_df['Unique Detection Time (UTC)'], format = '%Y-%m-%d %H:%M:%S')
    
    return unmatched_dates

def process_single_folder(full_path, folder_name, output_plot_directory, h5_file_path, unmatched_dates):
    folder_path = os.path.join(full_path, folder_name)
    try:
        npy_files = [np.load(os.path.join(folder_path, file)) for file in os.listdir(folder_path) if file.endswith('.npy')]
        folder_data = np.concatenate(npy_files, axis=0)

        # Read timestamps from the .h5 file
        with h5py.File(h5_file_path, 'r') as h5_file:
            timestamps = np.array(h5_file['timestamps'])
            timestamps = timestamps.astype(int)  # Explicitly convert to int
            time_utc = convert_timestamps_to_utc(timestamps)
    except Exception as e:
        print(f"Error processing folder {folder_path}: {e}")
        return
    
    detection_sig, mad = calculate_detection_sig(folder_data)
    print(1/mad)
    

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12), gridspec_kw={'height_ratios': [1, 1]})

    plot_time_series(ax1, time_utc[:len(folder_data)], folder_data, mad, folder_name, unmatched_dates)
    plot_histogram(ax2, detection_sig, folder_name)
    
    

    fig.tight_layout()
    plot_filename = os.path.join(output_plot_directory, f'plot_{folder_name}.png')
    print(output_plot_directory)
    plt.savefig(plot_filename, dpi=300)
    plt.show()
    #plt.close(fig)

# Example usage
output_plot_directory = os.path.join('.', 'plots')
if not os.path.exists(output_plot_directory):
    os.makedirs(output_plot_directory)

folder_name = '2023-08-27_10-10-23.770'  # Reemplaza con el nombre de la carpeta que deseas procesar
h5_file_path = '/data/data4/veronica-scratch-rainier/swarm_august2023/results_CC_TMA/h5_files_timestamps/timestamps_2023-08-25_2023-08-31_00.00.00.h5'  # Ruta al archivo .h5 que contiene los timestamps
unmatched_file_path = '/home/velgueta/notebooks/project_Mt-Rainier_DAS/txtfiles/template_5sec_csv_results-2023-08-25_00.00-2023-08-31_00.00/resulting_detections.csv'

unmatched_dates = read_dataframe(unmatched_file_path)

process_single_folder(full_path, folder_name, output_plot_directory, h5_file_path, unmatched_dates)
